In [ ]:
# Setup input parameters
from datetime import datetime as dt
dbutils.widgets.text('bg_loadtimestamp', '')
bg_loadtimestamp = dbutils.widgets.get('bg_loadtimestamp')
bg_loadtimestamp_dt = dt.strptime(bg_loadtimestamp, '%Y-%m-%d %H:%M:%S.%f')
if not bg_loadtimestamp:
    bg_loadtimestamp = 'CAST(NULL AS Timestamp)'
else:
    bg_loadtimestamp = f"CAST('{bg_loadtimestamp}' AS Timestamp)"




In [ ]:
# Setup logging
def info(statement_name, target_object_database_name, target_object_schema_name, target_object_name, message):
    log_message('INFO', statement_name, target_object_database_name, target_object_schema_name, target_object_name, message)

def error(statement_name, target_object_database_name, target_object_schema_name, target_object_name, message):
    log_message('ERROR',statement_name, target_object_database_name, target_object_schema_name, target_object_name, message)

def log_message(log_level, statement_name, target_object_database_name, target_object_schema_name, target_object_name, message):
    log_df = spark.createDataFrame([(dt.now(), log_level, 'loader', 'MB_Databricks_Stage_JDBCGenerator', '{loadcontrol#loadcontrol#application_name}', '{loadcontrol#loadcontrol#application_environment_name}', bg_loadtimestamp_dt, statement_name, 'STG_ST_stores_Loader', target_object_database_name, target_object_schema_name, target_object_name,  message)], ['log_timestamp', 'log_level', 'execution_unit', 'project_name', 'application_name', 'application_environment_name', 'load_timestamp', 'statement_name', 'task_name', 'target_object_database_name', 'target_object_schema_name', 'target_object_name', 'message'])
    log_df.write.format('delta').mode('append').saveAsTable('`{loadcontrol#loadcontrol#database_name}`.`{loadcontrol#loadcontrol#schema_name}`.`{loadcontrol#loadcontrol#log_table_name}`')
    print(f"{dt.now().strftime('%Y/%m/%d, %H:%M:%S')} - {target_object_name}: {message}")




In [ ]:
# StageLoader: stores_Stage Loader_1

try:

    from pyspark.sql.functions import input_file_name

    from pyspark.sql.functions import col

    df = spark.read.jdbc('{mb_databricks_stage_jdbcgenerator#bigenius-source-server.database.windows.net#jdbc_url}', 'dbo.stores', properties={'user': {mb_databricks_stage_jdbcgenerator#bigenius-source-server.database.windows.net#jdbc_user}, 'password': {mb_databricks_stage_jdbcgenerator#bigenius-source-server.database.windows.net#jdbc_password}, 'driver':'{mb_databricks_stage_jdbcgenerator#bigenius-source-server.database.windows.net#jdbc_driver}'})
    df.createOrReplaceTempView("`stg_st_stores_source_temp`")

    spark.sql(f"""CREATE OR REPLACE TEMPORARY VIEW `stg_st_stores_source`
    AS
    SELECT
         CAST(NULL AS Timestamp) AS `bg_loadtimestamp`
        ,CAST(NULL AS String) AS `bg_sourcesystem`
        ,`s1`.`store_id` AS `store_id`
        ,`s1`.`store_name` AS `store_name`
        ,`s1`.`web_address` AS `web_address`
        ,`s1`.`physical_address` AS `physical_address`
        ,`s1`.`latitude` AS `latitude`
        ,`s1`.`longitude` AS `longitude`
        ,`s1`.`logo` AS `logo`
        ,`s1`.`logo_mime_type` AS `logo_mime_type`
        ,`s1`.`logo_filename` AS `logo_filename`
        ,`s1`.`logo_charset` AS `logo_charset`
        ,`s1`.`logo_last_updated` AS `logo_last_updated`
    FROM `stg_st_stores_source_temp` AS `s1`
    """)

    operation_metrics_collection = {}
    result_df = spark.sql(f"""
    INSERT
    OVERWRITE `{mb_databricks_stage_jdbcgenerator#stage#database_name}`.`{mb_databricks_stage_jdbcgenerator#stage#schema_name}`.`stg_st_stores` (
         `bg_loadtimestamp`
        ,`bg_sourcesystem`
        ,`store_id`
        ,`store_name`
        ,`web_address`
        ,`physical_address`
        ,`latitude`
        ,`longitude`
        ,`logo`
        ,`logo_mime_type`
        ,`logo_filename`
        ,`logo_charset`
        ,`logo_last_updated`
    )
    SELECT
         {bg_loadtimestamp} AS `bg_loadtimestamp`
        ,`bg_source`.`bg_sourcesystem` AS `bg_sourcesystem`
        ,`bg_source`.`store_id` AS `store_id`
        ,`bg_source`.`store_name` AS `store_name`
        ,`bg_source`.`web_address` AS `web_address`
        ,`bg_source`.`physical_address` AS `physical_address`
        ,`bg_source`.`latitude` AS `latitude`
        ,`bg_source`.`longitude` AS `longitude`
        ,`bg_source`.`logo` AS `logo`
        ,`bg_source`.`logo_mime_type` AS `logo_mime_type`
        ,`bg_source`.`logo_filename` AS `logo_filename`
        ,`bg_source`.`logo_charset` AS `logo_charset`
        ,`bg_source`.`logo_last_updated` AS `logo_last_updated`
    FROM `stg_st_stores_source` AS `bg_source`
    """)
    RowCountInserted = result_df.select("num_inserted_rows").collect()[0][0]
    pandas_df = result_df.toPandas()
    operation_metrics = pandas_df.to_dict(orient='records')
    operation_metrics_collection['reloadtarget_{mb_databricks_stage_jdbcgenerator#stage#database_name}_{mb_databricks_stage_jdbcgenerator#stage#schema_name}_stg_st_stores'] = operation_metrics
    info('ReloadTarget', '{mb_databricks_stage_jdbcgenerator#stage#database_name}', '{mb_databricks_stage_jdbcgenerator#stage#schema_name}', 'STG_ST_stores', str(operation_metrics))

except Exception as e:
    error('', '', '', '', str(e))
    raise



In [ ]:
dbutils.notebook.exit(operation_metrics_collection)
